In [1]:
from google.colab import files
uploaded = files.upload()



Saving Invistico_Airline.csv to Invistico_Airline.csv


In [4]:
import pandas as pd

df = pd.read_csv("Invistico_Airline.csv")

df.head()
df.info()
df["satisfaction"].value_counts()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129880 entries, 0 to 129879
Data columns (total 22 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   satisfaction                       129880 non-null  object 
 1   Customer Type                      129880 non-null  object 
 2   Age                                129880 non-null  int64  
 3   Type of Travel                     129880 non-null  object 
 4   Class                              129880 non-null  object 
 5   Flight Distance                    129880 non-null  int64  
 6   Seat comfort                       129880 non-null  int64  
 7   Departure/Arrival time convenient  129880 non-null  int64  
 8   Food and drink                     129880 non-null  int64  
 9   Gate location                      129880 non-null  int64  
 10  Inflight wifi service              129880 non-null  int64  
 11  Inflight entertainment             1298

,0
satisfaction,0
Customer Type,0
Age,0
Type of Travel,0
Class,0
Flight Distance,0
Seat comfort,0
Departure/Arrival time convenient,0
Food and drink,0
Gate location,0


In [5]:
df["satisfaction"] = df["satisfaction"].map({
    "satisfied": 1,
    "dissatisfied": 0
})

In [6]:
X = df.drop("satisfaction", axis=1)
y = df["satisfaction"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [12]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

In [13]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [15]:
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

In [16]:
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Age', 'Flight Distance', 'Seat comfort',
       'Departure/Arrival time convenient', 'Food and drink', 'Gate location',
       'Inflight wifi service', 'Inflight entertainment', 'Online support',
       'Ease of Online booking', 'On-board service', 'Leg room...
       'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Customer Type', 'Type of Travel', 'Class'], dtype='object'))])),
                ('clf', LogisticRegression(max_iter=1000))])

In [17]:
from sklearn.preprocessing import StandardScaler

In [18]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [19]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [20]:
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        max_iter=3000,
        solver="lbfgs"
    ))
])

In [21]:
LogisticRegression(
    max_iter=3000,
    solver="saga",
    n_jobs=-1
)

LogisticRegression(max_iter=3000, n_jobs=-1, solver='saga')

In [22]:
LogisticRegression(max_iter=3000, n_jobs=-1, solver="saga")

LogisticRegression(max_iter=3000, n_jobs=-1, solver='saga')

In [23]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Age', 'Flight Distance', 'Seat comfort',
       'Departure/Arrival time convenient', 'Food and drink', 'Gate location',
       'Inflight wifi service', 'Inflight entertainment', 'Online support',
       'Ease of Online booking...
       'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Customer Type', 'Type of Travel', 'Class'], dtype='object'))])),
                ('clf', LogisticRegression(max_iter=3000))])

In [24]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [25]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm

array([[ 9544,  2215],
       [ 2233, 11984]])

In [26]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.81      0.81     11759
           1       0.84      0.84      0.84     14217

    accuracy                           0.83     25976
   macro avg       0.83      0.83      0.83     25976
weighted avg       0.83      0.83      0.83     25976



In [27]:
import pandas as pd

feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
}).sort_values("coef", ascending=False)

importance.head(10)
importance.tail(10)

,feature,coef
6,num__Inflight wifi service,-0.127863
0,num__Age,-0.133204
1,num__Flight Distance,-0.181496
17,num__Arrival Delay in Minutes,-0.258479
23,cat__Class_Eco,-0.293382
4,num__Food and drink,-0.296607
3,num__Departure/Arrival time convenient,-0.327559
24,cat__Class_Eco Plus,-0.349142
21,cat__Type of Travel_Personal Travel,-0.486751
19,cat__Customer Type_disloyal Customer,-1.046129


In [28]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
cm

array([[ 9544,  2215],
       [ 2233, 11984]])

In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.81      0.81     11759
           1       0.84      0.84      0.84     14217

    accuracy                           0.83     25976
   macro avg       0.83      0.83      0.83     25976
weighted avg       0.83      0.83      0.83     25976



In [30]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_prob)

np.float64(0.9035314775200614)

In [31]:
import pandas as pd

feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
}).sort_values("coef", ascending=False)

importance

,feature,coef
7,num__Inflight entertainment,0.972683
18,cat__Customer Type_Loyal Customer,0.826756
22,cat__Class_Business,0.423151
2,num__Seat comfort,0.392571
10,num__On-board service,0.388175
13,num__Checkin service,0.354962
9,num__Ease of Online booking,0.333694
11,num__Leg room service,0.308907
20,cat__Type of Travel_Business travel,0.267379
15,num__Online boarding,0.181688


In [32]:
importance.head(10)
importance.tail(10)

,feature,coef
6,num__Inflight wifi service,-0.127863
0,num__Age,-0.133204
1,num__Flight Distance,-0.181496
17,num__Arrival Delay in Minutes,-0.258479
23,cat__Class_Eco,-0.293382
4,num__Food and drink,-0.296607
3,num__Departure/Arrival time convenient,-0.327559
24,cat__Class_Eco Plus,-0.349142
21,cat__Type of Travel_Personal Travel,-0.486751
19,cat__Customer Type_disloyal Customer,-1.046129
